# Adversarial RF Robustness — Google Colab Runner

Upload the entire `adversarial-rf-robustness/` folder to Google Drive, then run this notebook with a GPU runtime.

**Runtime > Change runtime type > T4 GPU (or better)**

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where you uploaded the project
PROJECT_DIR = '/content/drive/MyDrive/ew project/adversarial-rf-robustness'

import os
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# 2. Install dependencies
!pip install scipy pandas matplotlib h5py pyyaml tqdm

In [ ]:
# 3. Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    #print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# 4. Run smoke test
!python smoke_test.py --data_path data/raw/RML2016.10a_dict.pkl

In [ ]:
# 5. Phase 0: Baseline CNN Training (~5 min on T4)
!python train.py \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --dataset_version 2016.10a \
  --epochs 60 \
  --batch_size 256 \
  --lr 0.001 \
  --seed 42 \
  --save_dir experiments/results/baseline_cnn \
  --device auto

In [ ]:
# 6. Phase 1: Clean Accuracy vs SNR
!python evaluate.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --eval_mode clean_snr \
  --num_trials 10 \
  --output_dir experiments/results/baseline_cnn \
  --device auto \
  --seed 42

In [ ]:
# 7. Phase 2: Attack evaluations (FGSM + PGD across channels)
for attack in ['fgsm', 'pgd']:
    for channel in ['awgn', 'rayleigh_awgn', 'rayleigh_cfo_awgn']:
        print(f'\n--- {attack.upper()} / {channel} ---')
        !python evaluate.py \
          --model_path experiments/results/baseline_cnn/best_model.pth \
          --data_path data/raw/RML2016.10a_dict.pkl \
          --eval_mode attack_snr \
          --attack_type {attack} \
          --channel_type {channel} \
          --rho 0.01 \
          --pgd_steps 10 \
          --num_trials 10 \
          --output_dir experiments/results/baseline_cnn \
          --device auto \
          --seed 42

In [ ]:
# 8. Phase 2: Attack vs Perturbation Budget
for attack in ['fgsm', 'pgd']:
    print(f'\n--- Budget sweep: {attack.upper()} ---')
    !python evaluate.py \
      --model_path experiments/results/baseline_cnn/best_model.pth \
      --data_path data/raw/RML2016.10a_dict.pkl \
      --eval_mode attack_budget \
      --attack_type {attack} \
      --channel_type awgn \
      --pgd_steps 10 \
      --num_trials 10 \
      --output_dir experiments/results/baseline_cnn \
      --device auto \
      --seed 42

In [ ]:
# 9. Phase 3: Defense Training & Evaluation (~20 min on T4)
!python train_defenses.py \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --epochs 60 \
  --batch_size 256 \
  --lr 0.001 \
  --seed 42 \
  --rho 0.01 \
  --pgd_steps 5 \
  --sigma 0.01 \
  --output_dir experiments/results \
  --eval_snr 0 10 \
  --device auto

In [ ]:
# 10. Phase 4: Per-Class Vulnerability Analysis
!python eval_per_class.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --snr 0 10 18 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results \
  --device auto

In [ ]:
# 11. Generate Figures
!python plot_results.py \
  --results_dir experiments/results \
  --output_dir experiments/figures

In [ ]:
# 12. View results
import pandas as pd

print('=== Defense Comparison ===')
df = pd.read_csv('experiments/results/defense_comparison.csv')
print(df.to_string(index=False))

print('\n=== Per-Class Vulnerability (SNR=10) ===')
df2 = pd.read_csv('experiments/results/per_class_vulnerability.csv')
df2_10 = df2[df2['snr'] == 10].sort_values('asr', ascending=False)
print(df2_10[['modulation', 'clean_acc', 'robust_acc', 'asr']].to_string(index=False))

In [ ]:
# 13. Display figures
from IPython.display import Image, display
import glob

for fig_path in sorted(glob.glob('experiments/figures/*.png')):
    print(f'\n{os.path.basename(fig_path)}')
    display(Image(filename=fig_path, width=600))